In [1]:
import openai
import pandas as pd

In [12]:
# split data into chunks of 9500 rows and save each chunk to a separate CSV file
def save_chunks_to_csv(df, chunk_size=9500):
    chunks = split_dataframe(df, chunk_size)
    for i, chunk in enumerate(chunks):
        chunk.to_csv(f'data_chunk_{i + 1}.csv', index=False)    
def split_dataframe(df, chunk_size=9500):
    return [df[i:i + chunk_size] for i in range(0, df.shape[0], chunk_size)]
save_chunks_to_csv(data, chunk_size=9500)

In [ ]:
# Inisialisasi client OpenAI
client = openai.OpenAI(api_key="openai-api-key")

def classify_sentiment(text):
    prompt = f"""
Kamu adalah model analisis sentimen. Tugasmu adalah mengklasifikasikan tweet tentang layanan Indihome ke dalam salah satu dari tiga kategori: Positif, Negatif, atau Netral.

Aturan klasifikasi:

Pilih Positif jika tweet menunjukkan pujian, kepuasan, atau opini baik terhadap layanan Indihome.

Pilih Negatif jika tweet menunjukkan keluhan, kekecewaan, atau opini buruk terhadap layanan Indihome.

Pilih Netral jika tweet bersifat informatif, bertanya, atau tidak menunjukkan opini yang jelas.

Berikan hanya satu label: "Positif", "Negatif", atau "Netral" tanpa penjelasan tambahan.
Tweet:
\"\"\"{text}\"\"\"
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano-2025-04-14", # ganti gpt-4.1-nano-2025-04-14 atau gpt-4o-mini-2024-07-18
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        label = response.choices[0].message.content.strip().capitalize()
        if label not in ["Positif", "Negatif", "Netral"]:
            return "Netral"
        return label
    except Exception as e:
        print(f"Error klasifikasi sentiment: {e}")
        return "Netral"

def classify_chats_sentiment(df):
    total = len(df)
    df = df.copy()
    df["Sentiment"] = None

    for i in range(total):
        teks = df.at[i, "full_text"]
        if pd.isna(teks) or not teks.strip():
            df.at[i, "Sentiment"] = "Netral"
        else:
            df.at[i, "Sentiment"] = classify_sentiment(teks)

        if (i + 1) % 100 == 0:
            print(f"Memproses {i+1}/{total} baris...")

    return df

if __name__ == "__main__":
    df = pd.read_csv('data_chunk_8.csv')
    df = classify_chats_sentiment(df)
    df.to_csv('data_chunk_8_with_sentiment.csv', index=False)
    print("Selesai klasifikasi sentimen!")

Memproses 100/9500 baris...
Memproses 200/9500 baris...
Memproses 300/9500 baris...
Memproses 400/9500 baris...
Memproses 500/9500 baris...
Memproses 600/9500 baris...
Memproses 700/9500 baris...
Memproses 800/9500 baris...
Memproses 900/9500 baris...
Memproses 1000/9500 baris...
Memproses 1100/9500 baris...
Memproses 1200/9500 baris...
Memproses 1300/9500 baris...
Memproses 1400/9500 baris...
Memproses 1500/9500 baris...
Memproses 1600/9500 baris...
Memproses 1700/9500 baris...
Memproses 1800/9500 baris...
Memproses 1900/9500 baris...
Memproses 2000/9500 baris...
Memproses 2100/9500 baris...
Memproses 2200/9500 baris...
Memproses 2300/9500 baris...
Memproses 2400/9500 baris...
Memproses 2500/9500 baris...
Memproses 2600/9500 baris...
Memproses 2700/9500 baris...
Memproses 2800/9500 baris...
Memproses 2900/9500 baris...
Memproses 3000/9500 baris...
Memproses 3100/9500 baris...
Memproses 3200/9500 baris...
Memproses 3300/9500 baris...
Memproses 3400/9500 baris...
Memproses 3500/9500 bar

In [5]:
df['Sentiment'].value_counts()

Sentiment
Netral     6902
Negatif    2305
Positif     293
Name: count, dtype: int64

In [7]:
df['Sentiment'].value_counts()

Sentiment
Netral     7021
Negatif    2214
Positif     265
Name: count, dtype: int64

In [10]:
for index, row in df.iterrows():
    if row['Sentiment'] == 'Positif':
        print(f"Index: {index}, Sentiment: {row['Sentiment']}, Text: {row['full_text']}")

Index: 48, Sentiment: Positif, Text: @minheoe Makasih udah menggunakan IndiHome Kak Udah berlangganan dari kapan nih? -Tammy
Index: 144, Sentiment: Positif, Text: @iconicholo @jaldeuroga Yuhuuu mantul banget emang gak perlu diragukan kalau pakai IndiHome. -Anggi
Index: 217, Sentiment: Positif, Text: Rayakan Keseruan Idulfitri-mu dengan IndiHome! Nikmati momen berharga dan silaturahmi bersama keluarga di rumah dengan koneksi internet stabil IndiHome. #IndiHomebyTelkomsel #DariRumahTanpaBatas https://t.co/k1i6vfhKyY
Index: 219, Sentiment: Positif, Text: @Galih_RN29 @IndiHome @Galih_RN29 Baik Kak Galih segera dibantu pengecekan lebih lanjut via DM @IndiHome yaa :) -Frey
Index: 344, Sentiment: Positif, Text: @SahawehEAN Mantappp moga tetap aman dan nyaman terus aktifitas bareng IndiHome-nya ya Kak . -Abdul
Index: 364, Sentiment: Positif, Text: @desidle ka kyknya emang indihome soalnya aku jugaaa ini pake data lancar
Index: 421, Sentiment: Positif, Text: @IndiHome Gapapa kak sudah baik
Inde

In [40]:
df['Sentiment'].value_counts()

Sentiment
Netral     5277
Negatif    3850
Positif     373
Name: count, dtype: int64